# Data Cleaning Workflow

This notebook starts the reproducible data-cleaning workflow for the selected target and predictors. Reusable download logic lives in `2_data/scripts/raw_data_download.py`; this notebook only imports the script, runs the workflow, and displays the resulting manifest.

## Optional Colab Setup

This cell is safe for local use. It only performs setup when the notebook runs inside Google Colab.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Colab detected. Mount your repository and set ROOT_DIR in the next cell if needed.")
else:
    print("Local environment detected; skipping Colab setup.")

## Repository Setup

In [1]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "0_organization").exists() and (path / "2_data").exists():
            return path
    raise RuntimeError("Could not find repository root. Run this notebook from inside the repository.")


ROOT_DIR = find_repo_root(Path.cwd())
SCRIPT_DIR = ROOT_DIR / "2_data" / "scripts"

if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

ROOT_DIR

PosixPath('/Users/marcjin/Desktop/Studium/Master/26SS/env-innovation-prediction')

## Raw Download Plan

The plan includes the finalized main target, the robustness target, the main predictors, and the policy robustness predictor.

In [2]:
import pandas as pd
from raw_data_download import build_raw_download_plan, run_raw_download

START_YEAR = 1990
END_YEAR = 2024

download_plan = pd.DataFrame(build_raw_download_plan(START_YEAR, END_YEAR))
download_plan[["dataset_id", "variable", "source_variable", "role", "start_year", "end_year"]]

,dataset_id,variable,source_variable,role,start_year,end_year
0,oecd_patents_environment,env_patent_share_inventions,PT_INV.DEV.ENV_PAT._Z,main_target,1990,2023
1,oecd_patents_environment,env_patents_per_million,INV_PS.DEV.ENV_PAT._Z,robustness_target,1990,2023
2,world_bank_wdi,gdp_per_capita,NY.GDP.PCAP.KD,main_predictor,1990,2024
3,world_bank_wdi,rd_expenditure_gdp,GB.XPD.RSDV.GD.ZS,main_predictor,1990,2024
4,world_bank_wdi,renewable_energy_share,EG.FEC.RNEW.ZS,main_predictor,1990,2024
5,oecd_eps,eps_index,POL_STRINGENCY.EPS,policy_robustness_predictor,1990,2020


## Download Raw Files

Running this cell writes source files to `2_data/raw/` and writes `raw_download_manifest.csv` with source URLs and file paths.

In [3]:
manifest = run_raw_download(start_year=START_YEAR, end_year=END_YEAR)
manifest[["dataset_id", "variable", "role", "rows", "columns", "file_path"]]

,dataset_id,variable,role,rows,columns,file_path
0,oecd_patents_environment,env_patent_share_inventions,main_target,3804,26,/Users/marcjin/Desktop/Studium/Master/26SS/env...
1,oecd_patents_environment,env_patents_per_million,robustness_target,3727,26,/Users/marcjin/Desktop/Studium/Master/26SS/env...
2,world_bank_wdi,gdp_per_capita,main_predictor,9310,7,/Users/marcjin/Desktop/Studium/Master/26SS/env...
3,world_bank_wdi,rd_expenditure_gdp,main_predictor,9310,7,/Users/marcjin/Desktop/Studium/Master/26SS/env...
4,world_bank_wdi,renewable_energy_share,main_predictor,9310,7,/Users/marcjin/Desktop/Studium/Master/26SS/env...
5,oecd_eps,eps_index,policy_robustness_predictor,1240,22,/Users/marcjin/Desktop/Studium/Master/26SS/env...


## Next Step

The next cleaning step should read these raw files, standardize country-year columns, construct three-year lagged moving averages for predictors, and write a processed modeling panel to `2_data/processed/`.